# Hands-on Lab: Analyzing Historical Stock/Revenue Data and Building a Dashboard

This notebook collects Tesla (TSLA) and GameStop (GME) stock data with `yfinance`, scrapes their quarterly revenue with `requests` + BeautifulSoup, and builds a dashboard of interactive Plotly graphs.

**Run all cells from top to bottom (Kernel > Restart & Run All), then save before uploading.**

In [1]:
%pip install yfinance beautifulsoup4 lxml html5lib plotly pandas requests nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

## Define the graphing function

In [3]:
def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()

    # Make dates comparable (timezone-naive datetimes)
    stock_data["Date"] = pd.to_datetime(stock_data["Date"], utc=True).dt.tz_localize(None)
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"])

    stock_data_specific = stock_data[stock_data["Date"] <= "2021-06-14"]
    revenue_data_specific = revenue_data[revenue_data["Date"] <= "2021-04-30"]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.25,
    )
    fig.add_trace(go.Scatter(x=stock_data_specific["Date"],
                             y=stock_data_specific["Close"].astype("float"),
                             name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data_specific["Date"],
                             y=revenue_data_specific["Revenue"].astype("float"),
                             name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", showticklabels=True, type="date", row=1, col=1)
    fig.update_xaxes(title_text="Date", showticklabels=True, type="date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock,
                      xaxis_rangeslider_visible=False)
    fig.show()

## Helper: scrape a quarterly revenue table with BeautifulSoup

In [4]:
def scrape_revenue(url):
    html_data = requests.get(url).text
    soup = BeautifulSoup(html_data, "html.parser")

    # Find the table holding the quarterly revenue data
    tables = soup.find_all("table")
    table = next((t for t in tables if "Revenue" in t.get_text()), tables[1])

    rows = []
    for tr in table.find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) == 2:
            rows.append({"Date": cells[0].get_text(strip=True),
                         "Revenue": cells[1].get_text(strip=True)})

    df = pd.DataFrame(rows, columns=["Date", "Revenue"])

    # Remove the $ symbol and commas, then drop empty rows
    df["Revenue"] = df["Revenue"].str.replace(r"[$,]", "", regex=True)
    df.dropna(inplace=True)
    df = df[df["Revenue"] != ""].reset_index(drop=True)
    return df

## Question 1 - Extracting Tesla Stock Data Using yfinance

In [5]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Question 2 - Extracting Tesla Revenue Data Using Webscraping

In [6]:
tesla_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = scrape_revenue(tesla_url)
tesla_revenue.tail()

,Date,Revenue
8,2013,2013
9,2012,413
10,2011,204
11,2010,117
12,2009,112


## Question 3 - Extracting GameStop Stock Data Using yfinance

In [7]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620129,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666417,1.666417,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


## Question 4 - Extracting GameStop Revenue Data Using Webscraping

In [8]:
gme_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = scrape_revenue(gme_url)
gme_revenue.tail()

,Date,Revenue
11,2009,8806
12,2008,7094
13,2007,5319
14,2006,3092
15,2005,1843


## Question 5 - Tesla Stock and Revenue Dashboard

In [9]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 - GameStop Stock and Revenue Dashboard

In [10]:
make_graph(gme_data, gme_revenue, "GameStop")